# Create a graph for the possible future road network
For each component of stands that is not connected to any roads (respective: big roads), we wanna create a network of possible future road segments which strictly follow the boundaries of the forest stands.

This notebook builds and refines graph representations of components, preparing them for later optimization steps.  
Each component is processed independently, and results are saved in component-specific folders.  

For each component (`name`), the workflow performs the following steps:

1. **Setup & Extraction**  
   - Create a dedicated output folder for the component.  
   - Extract **boundary points, edges, attributes, exit points, and stand-to-node mappings**.  

2. **Initial Graph Construction & Visualization**  
   - Build the initial graph from extracted data.  
   - Plot and save the graph for inspection.  

3. **Exit Points Handling**  
   - Identify existing exit points and add missing ones.  
   - Update and save the graph with exit nodes integrated.  

4. **Assigning Edge Costs**  
   - Assign cost values to each edge based on attributes.  
   - Save and visualize the cost-enhanced graph.  

5. **Edge Merging**  
   - Merge short edges below a given length threshold (default: 2000).  
   - Save and visualize the simplified graph.  
   - Verify the merge process for consistency.  

6. **Source Nodes & Imaginary Edges**  
   - Add centroids (representing source nodes) and connect them with imaginary edges.  
   - Assign zero costs to imaginary edges.  
   - Save and visualize the updated graph.  

7. **Final Data Storage**  
   - Save graph data (nodes, arcs, boundaries) for downstream use.  
   - Ensure the component’s representation is complete and ready for optimization.  

This workflow ensures that each component is transformed from raw boundaries and attributes into a **clean, cost-assigned, and source/exit-augmented graph** representation.


## Imports & Settings

In [1]:
# Import standard libraries
import os
import shutil
import re
from pathlib import Path
import csv
import json
import math
from collections import Counter
from glob import glob

# Import third-party libraries
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import missingno as msno

# Import geometrical and spatial libraries
from shapely.geometry import MultiPolygon, Polygon, Point, LineString
from shapely import wkt

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib_scalebar.scalebar import ScaleBar


## Load roads

In [2]:
# Load the suitable roads
roads = gpd.read_file(r"1_Preprocessed_Data/1_Roads_clean/roads_min_3m.shp")

# Check CRS alignment for roads (optional)
print(f"CRS of roads: {roads.crs}")

# Print the number of roads loaded and preview the first row
print(f"{len(roads)} usable roads loaded")
roads.head(1)

CRS of roads: PROJCRS["ETRS89 / Portugal TM06",BASEGEOGCRS["ETRS89",DATUM["European Terrestrial Reference System 1989",ELLIPSOID["GRS 1980",6378137,298.257222101,LENGTHUNIT["metre",1]],ID["EPSG",6258]],PRIMEM["Greenwich",0,ANGLEUNIT["Degree",0.0174532925199433]]],CONVERSION["unnamed",METHOD["Transverse Mercator",ID["EPSG",9807]],PARAMETER["Latitude of natural origin",39.6682583333333,ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG",8801]],PARAMETER["Longitude of natural origin",-8.13310833333333,ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG",8802]],PARAMETER["Scale factor at natural origin",1,SCALEUNIT["unity",1],ID["EPSG",8805]],PARAMETER["False easting",0,LENGTHUNIT["metre",1],ID["EPSG",8806]],PARAMETER["False northing",0,LENGTHUNIT["metre",1],ID["EPSG",8807]]],CS[Cartesian,3],AXIS["(E)",east,ORDER[1],LENGTHUNIT["metre",1,ID["EPSG",9001]]],AXIS["(N)",north,ORDER[2],LENGTHUNIT["metre",1,ID["EPSG",9001]]],AXIS["ellipsoidal height (h)",up,ORDER[3],LENGTHUNIT["metre",1,ID["EPSG",9001]]

,Id,ID_RV,DATA_ACCAO,COD_INE,DESIGNACAO,OPERAC,REDE_DFCI,TIPO_PISO,COMPRIM,LARGURA,...,FASE_2017,TIPO_VEICU,INTER_2018,EXEC_2018,FIN_2018,FASE_2018,OBSERV,LARGURA_ZE,ROADWIDTH,geometry
0,1,1,2011-01-31,10609,VARIANTE EN222,OPER,1,A,1398.32,0.0,...,0,VTTR,ESI,0,0,0,NaN,1,6.0,"LINESTRING (-11588.474 151775.913, -11614.929 ..."


## Load stands components

In [3]:
# Set input and output paths
inpath = r"1_Preprocessed_Data\3_Stand_Components\3_merged_components"
out_directory = r"1_Preprocessed_Data\4_Road_Network_Graphs"

In [4]:
# Initialize an empty dictionary to store the components
components = {}
total_n_stands = 0

# List all shapefiles in the directory
shapefiles = [f.path for f in os.scandir(inpath) if f.is_file() and f.name.endswith('.shp')]

# Load each shapefile into the dictionary
for shapefile_path in shapefiles:
    # Extract the filename without extension
    filename = os.path.splitext(os.path.basename(shapefile_path))[0].removesuffix('_stands')
    
    # Load the shapefile into a GeoDataFrame
    stands = gpd.read_file(shapefile_path)
    components[filename] = stands
    n_stands = len(stands)
    print(f"Component '{filename}' loaded with {len(stands)} stands.")
    total_n_stands+= n_stands
    print(total_n_stands)    

Component 'comp_10' loaded with 18 stands.
18
Component 'comp_14_15_48_merged' loaded with 62 stands.
80
Component 'comp_16' loaded with 14 stands.
94
Component 'comp_17' loaded with 1 stands.
95
Component 'comp_18' loaded with 3 stands.
98
Component 'comp_19_41_merged' loaded with 6 stands.
104
Component 'comp_1' loaded with 3 stands.
107
Component 'comp_20_25_28_33_merged' loaded with 56 stands.
163
Component 'comp_21_40_merged' loaded with 14 stands.
177
Component 'comp_22' loaded with 7 stands.
184
Component 'comp_23' loaded with 14 stands.
198
Component 'comp_24' loaded with 12 stands.
210
Component 'comp_26' loaded with 4 stands.
214
Component 'comp_27' loaded with 6 stands.
220
Component 'comp_29' loaded with 4 stands.
224
Component 'comp_2_44_merged' loaded with 87 stands.
311
Component 'comp_30' loaded with 3 stands.
314
Component 'comp_31_34_merged' loaded with 17 stands.
331
Component 'comp_32' loaded with 8 stands.
339
Component 'comp_35' loaded with 3 stands.
342
Component

In [5]:
# --- Concatenate all components into a single GeoDataFrame ---
all_stands = pd.concat(components.values(), ignore_index=True)
print(f"\nTotal number of rows (from all components together): {len(all_stands)}")
print(f"Number of unique ID_UGs (from all components together): {all_stands['ID_UG'].nunique()}")


Total number of rows (from all components together): 577
Number of unique ID_UGs (from all components together): 549


In [6]:
# --- Check for duplicates across all components ---
duplicates = all_stands[all_stands.duplicated(subset='ID_UG', keep=False)]

if len(duplicates) > 0:
    print(f"\nDuplicate ID_UGs found across all components ({len(duplicates)} rows):")
    # Sort by ID_UG to make duplicates more obvious
    duplicates_sorted = duplicates.sort_values(by='ID_UG').reset_index(drop=True)
    #print(duplicates_sorted[['ID_UG']])
else:
    print("\nNo duplicate ID_UGs found across all components.")



Duplicate ID_UGs found across all components (55 rows):


In [7]:
len(all_stands['ID_UG'].to_list())

577

In [8]:
example = components['comp_1']
example.head(1)

,OBJECTID,TARGET_FID,LandUse_1,Ocupacao,ID_UG,NOME,YY_correct,XX_Correct,Altitude,Declive,...,Hectares,UsoSolo20,Shape_Leng,Shape_Area,HBC,CH,CBD,CC,road_acces,geometry
0,128,758,1,EcEc_3_10_2_Pv,837,Castelo de Paiva,153027.146729,-21569.12586,151.0,11.6688,...,17.097563,Eucalipto,2076.935175,170975.628174,2.9,0.0,0.0,0.0,full,"POLYGON ((-21287.683 153143.277, -21333.710 15..."


In [9]:
example.crs

<Projected CRS: PROJCRS["ETRS89 / Portugal TM06",BASEGEOGCRS["ETRS ...>
Name: ETRS89 / Portugal TM06
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
- h[up]: Ellipsoidal height (metre)
Area of Use:
- undefined
Coordinate Operation:
- name: unnamed
- method: Transverse Mercator
Datum: European Terrestrial Reference System 1989
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

Projected CRS, Euclidean distance is in meters.

## 1. Extract vertices and edges with attributes (slope, edge length) and exit points from boundaries [helper functions]
From the boundaries of the stands, we extract all points with coordianates and additionally save the edges (lines) connecting them. 

Note: We only want need exterior boundaries, and NO roads along interior boundaries of a forest stand (because such interior roads would not help with connecting the stands to the existing road network).

There will be helper functions for step I consider important enough to mention them seperately, even if the function might be a single line of code, it helps to beter understand the structure and what happens.

### 1.0 Snap coordinates to grid [helper function] 
Before comparing edges, snap all coordinates to a common grid (round coordinates to a fixed number of decimal places). This helps ensure that neighboring stands with slightly different coordinate representations share identical coordinates.

In [10]:
def snap_to_grid(coord, precision):
    """
    Snaps the given coordinates to the specified precision.

    Parameters:
        coord (tuple or list): A tuple or list containing the x, y (and optionally z) coordinates to be snapped.
        precision (int): The number of decimal places to round the coordinates.

    Returns:
        tuple: A tuple of coordinates rounded to the specified precision.

    Raises:
        ValueError: If the input is not a tuple or list, or if any coordinate is NaN or infinite.
    """
    # Check if input is a tuple or list
    if not isinstance(coord, (tuple, list)):
        raise ValueError("Input must be a tuple or list of coordinates")

    # Check for NaN or infinity values
    if any(map(lambda c: isinstance(c, (float, int)) and (math.isnan(c) or math.isinf(c)), coord)):
        raise ValueError("Coordinate contains NaN or infinite values")
    
    # Round coordinates to the specified precision
    return tuple(round(c, precision) for c in coord)

### 1.1 Calculate attributes [helper function]
To calculate the costs, we need to know for each edge its edge length and its approximative slope.
- *edgelength*: The length of an edge (u,v) is calculated using the Euclidean distance formula, to compute the distance between the coordinates of points u and v.
- *slope (approx.)*: The slope of an edge is approximated by the slope ("Declive") of the polygon the edge belongs to.


In [11]:
# Check geometry dimension for the one example component
component = next(iter(components.values()))  # Get the first component
geometry = component.geometry.iloc[0]

if geometry.has_z:
    print("The shapefile contains 3D geometries with Z-values (altitudes per coordinate).")
else:
    print("The shapefile contains 2D geometries (no altitude per coordinate).")
    
    # Check for altitude attribute
    if 'Altitude' in component.columns:
        print("The shapefile has an 'Altitude' column (single altitude per polygon).")
    else: 
        print("No 'Altitude' column found.")

The shapefile contains 2D geometries (no altitude per coordinate).
The shapefile has an 'Altitude' column (single altitude per polygon).


Note: If we knew the altitude per coordinate, the slope could be approximated via the ratio of the altitude difference to the edgelength; but the edges all belong to the same polygon so we only have one single altitude per polygon (which would lead to a slope of 0). Therefore, we just take the slope ("Declive") of the polygon.

In [12]:
def calculate_edge_length(u, v):
    """
    Calculate the length of the edge defined by two vertices (u, v).

    Parameters:
        u (tuple): A tuple representing the coordinates (x, y) of the first vertex.
        v (tuple): A tuple representing the coordinates (x, y) of the second vertex.

    Returns:
        float: The length of the edge between vertices u and v.

    Raises:
        ValueError: If the input coordinates are not tuples or lists of length 2.
    """
    # Ensure the input coordinates are valid (tuples of length 2)
    if not (isinstance(u, (tuple, list)) and len(u) == 2) or not (isinstance(v, (tuple, list)) and len(v) == 2):
        raise ValueError("Both u and v must be tuples or lists of length 2 representing coordinates.")

    # Create a LineString object to calculate the edge length
    line = LineString([u, v])

    edgelength = line.length / 1000 #line.length gives length in meter because we use projected CRS. divide by 1000 because meter -> km

    # Return the length of the line
    return edgelength

### 1.2 Extract nodes, edges, attributes [helper function]

In [13]:
print("Multipolygon features:")
for name, df in components.items():
    multipolygons = df[df.geometry.geom_type == 'MultiPolygon']
    if not multipolygons.empty: print(name, multipolygons[['ID_UG', 'geometry']])

Multipolygon features:


In [14]:
def extract_boundaries_with_attributes(outpath, stands, precision=5):
    """
    Extracts the vertices, edges, edge attributes (such as edge length and slope),
    and exit points (intersections with roads) from polygon geometries in the input 
    stands dataset. Coordinates and intersections are snapped to a grid for precision.

    Parameters:
        outpath (string): Directory where to save results
        stands (GeoDataFrame): A GeoDataFrame containing the stand boundaries and 
                                associated data. Each feature must have a polygon geometry.
        precision (int, optional, default=4): The precision to which the coordinates 
                                              will be snapped when extracted (controls decimal places).

    Returns:
        tuple: A tuple containing five elements:
            - boundary points
            - boundary edges
            - edge_attributes (list of dicts): A list of dictionaries containing edge attributes (edge length, slope)
            - exit_points (list of tuples): A list of exit points (coordinates where stand boundaries intersect roads)
            - node_to_stands
    """
    boundary_points = set()  # Using set to avoid duplicate vertices
    boundary_edges = []
    edge_attributes = []
    exit_points = []
    node_to_stands = {}  # Dictionary to store which stands each node belongs to

    for _, feature in stands.iterrows():
        stand_id = feature['ID_UG']  # Unique identifier for the stand
        geometry = feature.geometry
        slope = feature['Declive']

        if geometry.geom_type == 'Polygon':
            # Snap coordinates for the exterior
            exterior_coords = [snap_to_grid(coord, precision) for coord in geometry.exterior.coords]
            boundary_points.update(exterior_coords)

            # Track which nodes belong to which stands
            for node in exterior_coords:#[:-1]:  # Ignore duplicate last point
                if node not in node_to_stands:
                    node_to_stands[node] = set()  # Use a set to avoid duplicates
                node_to_stands[node].add(stand_id)

            # Creating edges
            for i in range(len(exterior_coords) - 1):
                u, v = exterior_coords[i], exterior_coords[i + 1]
                boundary_edges.append((u, v))

                # Edge length and slope
                edge_length = calculate_edge_length(u, v)
                edge_attributes.append({'edgelength': edge_length,
                                        'slope': slope,
                                        'has_exit': False,
                                        'has_source': False})

            # Find intersections with roads (exit points)
            stand_boundary = geometry.exterior  # Stand polygon boundary
            for road in roads.geometry:
                if stand_boundary.intersects(road):  # Only proceed if there is an intersection
                    intersection = stand_boundary.intersection(road)
                    
                    # Process intersections
                    if intersection.geom_type == 'Point':
                        exit_points.append(snap_to_grid((intersection.x, intersection.y), precision))
                    elif intersection.geom_type == 'MultiPoint':
                        exit_points.extend([snap_to_grid((point.x, point.y), precision) for point in intersection.geoms])

            # Track which exitnodes belong to which stands (exit points)
            for exit_node in exit_points:  # For exit nodes only
                if exit_node not in node_to_stands:
                    node_to_stands[exit_node] = set()  # Use a set to avoid duplicates
                    node_to_stands[exit_node].add(stand_id)            

    # Convert sets to lists for easier graph storage
    node_to_stands = {node: list(stand_ids) for node, stand_ids in node_to_stands.items()} 

    # Save as CSV
    with open(f'{outpath}/node_to_stands.csv', 'w', newline='') as f:
        csv.writer(f).writerow(['Node', 'Stand'])  # Write header
        csv.writer(f).writerows(node_to_stands.items())  # Write data rows

    return list(boundary_points), boundary_edges, edge_attributes, exit_points, node_to_stands

## 2. Create the Graph
Next, we'll use these vertices and edges to build a graph. This graph will represent the potential road network, where the roads are constrained to follow the boundaries of the forest stands.

### 💾 Storing graph data [helper function] 

In [15]:
def store_data_for_graph(out_path, vertices, edges, attributes, name):
    """
    Saves the data for the graph (vertices, edges, and attributes) into CSV files within a specified folder.

    Parameters:
        out_path (str): The directory where the CSV files will be saved.
        vertices (list): A list of vertex coordinates (x, y) to be saved.
        edges (list): A list of edges, each represented by a tuple of vertices.
        attributes (list): A list of dictionaries containing edge attributes (e.g., 'edgelength', 'slope').
        name (str): The name of the component, used in the file naming.
        G (networkx.Graph): The graph containing node attributes, including `is_exit`.
    """

    # Ensure the folder exists
    os.makedirs(out_path, exist_ok=True)
    name=""

    # File paths
    vertices_file = os.path.join(out_path, f'nodes_{name}.csv')
    edges_file = os.path.join(out_path, f'edges_{name}.csv')
    attributes_file = os.path.join(out_path, f'attributes_{name}.csv')
    edges_with_attributes_file = os.path.join(out_path, f'edges_with_attributes_{name}.csv')

    # Save vertices to CSV with the 'is_exit' column
    vertices_data = [{'x': x, 'y': y, } for x, y in vertices]
    pd.DataFrame(vertices_data).to_csv(vertices_file, index=False)

    # Save edges to CSV
    edges_data = [(f"({u[0]}, {u[1]})", f"({v[0]}, {v[1]})") for u, v in edges]
    pd.DataFrame(edges_data, columns=['Node1(x,y)', 'Node2(x,y)']).to_csv(edges_file, index=False)

    # Save attributes to CSV
    pd.DataFrame(attributes).to_csv(attributes_file, index=False)

    # Save edges with attributes to CSV
    edges_attributes_data = [
        {
            'Node1(x,y)': f"({u[0]}, {u[1]})",
            'Node2(x,y)': f"({v[0]}, {v[1]})",
            'edgelength': attributes['edgelength'],
            'slope': attributes['slope'],
            'has_exit': attributes['has_exit'],
            'has_source': attributes['has_source']
        }
        for (u, v), attributes in zip(edges, attributes)
    ]
    pd.DataFrame(edges_attributes_data).to_csv(edges_with_attributes_file, index=False)


### Create graph and digraph from vertices, edges, attributes [helper function] 

In [16]:
def create_graph(vertices, edges, attributes, node_to_stands):
    """
    Create an undirected graph using NetworkX based on the provided vertices, edges, and edge attributes.

    Parameters:
        vertices (list): A list of vertices (nodes) to be added to the graph.
        edges (list): A list of edges represented as tuples (u, v), where u and v are vertices.
        attributes (list): A list of dictionaries, where each dictionary contains edge attributes 
                           (such as 'weight', 'length', 'has_exit', etc.) corresponding to each edge.

    Returns:
        nx.Graph: A NetworkX Graph object containing the nodes, edges, and associated attributes.
    
    Example:
        vertices = [(x1, y1), (x2, y2), (x3, y3)]
        edges = [((x1, y1), (x2, y2))]
        attributes = [
            {'edgelength': 74.92801170857655, 'slope': 18.0777, 'has_exit': False}
        ]
    """

    if len(edges) != len(attributes):
        raise ValueError("The number of edges and attributes must match.")

    G = nx.Graph()
    for node in vertices:
        G.add_node(node, is_exit=False, is_source=False, stands=node_to_stands.get(node, [])) 

    for edge, attr in zip(edges, attributes):
        u, v = edge
        # Add the edge with the attributes
        G.add_edge(u, v, **attr)
    
    return G

### Plot graph and store image [helper function]

In [17]:
def plot_and_save(outpath, G, name, prefix):
    """Plots the graph 'G' along with associated geographical components and saves the plot to a file.

    Parameters:
        G (nx.Graph): The NetworkX graph object to be plotted.
        name (str): The name of the component, used in the plot title and file naming.
        prefix (str): Prefix for the saved filename.

    Saves:
        A PNG image containing:
            - The geographical component from the 'components' GeoDataFrame.
            - The graph 'G' overlaid with nodes and edges.
            - A summary of node degrees, total edge length, and number of exit/source nodes as an annotation.
            - A legend for road access categories.
    """

    fig = plt.figure(figsize=(10, 10))
    filename = os.path.join(outpath, f'{prefix}_{name}.png')

    # Function to color GeoDataFrame polygons based on 'road_acces'
    def get_color(row):
        access = str(row.get('road_acces')).lower()
        if access == 'none':
            return (1.0, 0.8, 0.4, 0.5)  # transparent yellow-orange
        else:
            return (0.5, 1.0, 0.5, 0.5)  # transparent light green

    # Plot the GeoDataFrame with conditional colors
    components[name].plot(ax=plt.gca(), color=components[name].apply(get_color, axis=1))

    # Annotate each polygon with its ID_UG using road_acces-based text color
    def get_text_color(row):
        access = str(row.get('road_acces')).lower()
        if access == 'none':
            return (0.8, 0.4, 0.0)  # dense orange
        else:
            return (0.0, 0.0, 0.0)  # black for all other cases

    components[name].apply(
        lambda row: plt.annotate(
            text=row['ID_UG'],
            xy=(row.geometry.representative_point().x, row.geometry.representative_point().y),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=8,
            color=get_text_color(row)
        ),
        axis=1
    )

    # Prepare node colors
    nodecolor = [
        'orange' if G.nodes[node].get('is_source') else  # stronger orange-yellow for source nodes
        'blue' if G.nodes[node].get('is_exit') else
        'grey'  # black for regular nodes
        for node in G.nodes()
    ]

    # Prepare edge colors
    edgecolors = [
        'orange' if G[u][v].get('has_source') else 'grey'  # stronger orange-yellow for source edges
        for u, v in G.edges()
    ]

    nx.draw(
        G,
        pos={node: node for node in G.nodes()},
        node_size=50,
        node_color=nodecolor,
        edge_color=edgecolors,
        with_labels=False
    )

    # Node degree counts
    degree_counts = Counter(dict(G.degree()).values())

    # Total edge length
    total_edge_length = sum(G[u][v].get('edgelength', 0) for u, v in G.edges())

    # Calculate total costs if attributes exist
    cost_attributeskeys = ['Build5m', 'Maintain5m', 'Build10m', 'Maintain10m', 'Upgrade']
    costattributes_exist = all(
        any(attr in data for u, v, data in G.edges(data=True))
        for attr in cost_attributeskeys
    )

    if costattributes_exist:
        total_costs = sum(
            G[u][v].get('Build5', 0) +
            G[u][v].get('Maintain5', 0) +
            G[u][v].get('Build10', 0) +
            G[u][v].get('Maintain10', 0) +
            G[u][v].get('Upgrade', 0)
            for u, v in G.edges()
        )

    # Count source and exit nodes
    exit_nodes_count = sum(1 for node in G.nodes if G.nodes[node].get('is_exit'))
    source_nodes_count = sum(1 for node in G.nodes if G.nodes[node].get('is_source'))

    # Create overview text
    degree_overview = "Graph Degree Overview:\n"
    for degree, count in sorted(degree_counts.items()):
        degree_overview += f"Nodes with degree {degree}: {count}\n"
    degree_overview += f"\nNumber of Source Nodes: {source_nodes_count}\n"
    degree_overview += f"\nNumber of Exit Nodes: {exit_nodes_count}\n"
    degree_overview += f"Total Edge Length: {total_edge_length:.2f}\n"
    if costattributes_exist:
        degree_overview += f"Total Costs: {total_costs:.2f}\n"

    # Add title and text annotation
    plt.title(f"Graph Plot for {name},\n {len(G.nodes)} nodes, {len(G.edges)} edges")
    plt.text(0.5, -0.12, degree_overview, ha='center', va='top', transform=plt.gca().transAxes, fontsize=10)

    # --- Polygon legend (road access) ---
    polygon_legend = [
        mpatches.Patch(color=(1.0, 0.8, 0.4, 0.5), label='Inaccessible'),
        mpatches.Patch(color=(0.5, 1.0, 0.5, 0.5), label='Accessible')
    ]

     # --- Node/Edge legend ---
    graph_legend = [
        Line2D([0], [0], color='grey', lw=2, label='Edge'),
        Line2D([0], [0], marker='o', color='grey', label='Node',
            markerfacecolor='grey', markersize=8, linestyle='None'),
        Line2D([0], [0], marker='o', color='blue', label='Exit Node',
            markerfacecolor='blue', markersize=8, linestyle='None'),
        Line2D([0], [0], marker='o', color='orange', label='Source Node',
            markerfacecolor='orange', markersize=8, linestyle='None')  # stronger orange-yellow
    ]

    # Combine legends
    plt.legend(handles=polygon_legend + graph_legend, loc='upper left')

    # Save plot
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)


In [18]:
def save_graph_data_to_csv(outpath, G, name, prefix):
    """
    Calculates and appends graph summary statistics (as shown in the PNG annotation)
    into a CSV file stored one folder above 'outpath'.

    Parameters:
        outpath (str): Directory where PNGs are saved. CSV will go one level up.
        G (nx.Graph): NetworkX graph object.
        name (str): Name of the graph/component.
        prefix (str): Prefix for both PNG and CSV file naming.

    Appends:
        A CSV file in the parent directory of 'outpath' with the following columns:
            - Prefix
            - ComponentName
            - NumNodes
            - NumEdges
            - DegreeDistribution
            - NumSourceNodes
            - NumExitNodes
            - TotalEdgeLength
            - TotalCosts
    """
    # Ensure output directory exists
    os.makedirs(outpath, exist_ok=True)

    # Store CSV one folder above outpath
    parent_dir = os.path.dirname(outpath.rstrip("/\\"))
    csv_path = os.path.join(parent_dir, f"graph_summary.csv")

    # --- Compute metrics ---
    degree_counts = Counter(dict(G.degree()).values())
    total_edge_length = sum(G[u][v].get('edgelength', 0) for u, v in G.edges())

    exit_nodes_count = sum(1 for node in G.nodes if G.nodes[node].get('is_exit'))
    source_nodes_count = sum(1 for node in G.nodes if G.nodes[node].get('is_source'))

    # Cost attributes
    cost_attributes = ['Build5', 'Maintain5', 'Build10', 'Maintain10', 'Upgrade']
    costattributes_exist = any(attr in G[u][v] for u, v in G.edges() for attr in cost_attributes)
    total_costs = None
    if costattributes_exist:
        total_costs = sum(
            G[u][v].get('Build5', 0)
            + G[u][v].get('Maintain5', 0)
            + G[u][v].get('Build10', 0)
            + G[u][v].get('Maintain10', 0)
            + G[u][v].get('Upgrade', 0)
            for u, v in G.edges()
        )

    # Degree distribution as text
    degree_overview = "; ".join([f"{deg}:{cnt}" for deg, cnt in sorted(degree_counts.items())])

    # Data row
    data = {
        "Prefix": prefix,
        "ComponentName": name,
        "NumNodes": len(G.nodes),
        "NumEdges": len(G.edges),
        "DegreeDistribution": degree_overview,
        "NumSourceNodes": source_nodes_count,
        "NumExitNodes": exit_nodes_count,
        "TotalEdgeLength": round(total_edge_length, 2),
        "TotalCosts": round(total_costs, 2) if total_costs is not None else None
    }

    # --- Append to CSV (create with header if missing) ---
    file_exists = os.path.exists(csv_path)
    with open(csv_path, mode="a", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=data.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(data)

    print(f"✅ Graph summary for '{name}' appended to {csv_path}")


## 3. Handle exit points [helper functions]

### find nearest node [helper function]

In [19]:
def find_nearest_node(graph, exit_points):
    """
    Find the nearest nodes in the graph for a list of exit points based on Euclidean distance.

    Parameters:
        graph (nx.Graph): A NetworkX graph object containing nodes and their coordinates.
        exit_points (list): A list of exit points, where each exit point is a tuple representing (x, y) coordinates.

    Returns:
        pd.DataFrame: A DataFrame with columns ['exit_point', 'nearest_neighbor_node', 'distance'],
                      containing the exit points, their corresponding nearest nodes, and the calculated distances.

    Example:
        graph = nx.Graph()
        exit_points = [(x1, y1), (x2, y2)]
        result_df = find_nearest_node(graph, exit_points)
    """
    results = []
    for exit_point in exit_points:
        # Find the nearest node based on distance
        nearest_node = min(graph.nodes, key=lambda node: np.linalg.norm(np.array(exit_point) - np.array(node)))
        
        # Calculate the distance to the nearest node
        distance = np.linalg.norm(np.array(exit_point) - np.array(nearest_node))
        
        # Append the result
        results.append((exit_point, nearest_node, distance))
    
    # Create DataFrame
    df = pd.DataFrame(results, columns=['exit_point', 'nearest_neighbor_node', 'distance'])
    return df

### find edges closeby [helper function]

In [20]:
def find_closest_edge(G, exit_x, exit_y):
    """
    Finds the closest edge in the graph to the exit point (exit_x, exit_y).
    Returns the edge (u, v) and the intersection point.
    """
    min_dist = float('inf')
    closest_edge = None
    intersection_point = None

    # Iterate through all edges in the graph
    for u, v in G.edges():
        # Get the coordinates of the nodes (assumed to be stored as node attributes)
        ux, uy = u
        vx, vy = v
        
        # Create a LineString for the edge
        line = LineString([(ux, uy), (vx, vy)])

        # Create a Point for the exit
        exit_point = Point(exit_x, exit_y)

        # Check if the exit point intersects the edge
        if line.distance(exit_point) < min_dist:
            min_dist = line.distance(exit_point)
            closest_edge = (u, v)
            # If the point is on the line, we can get the intersection point
            if line.intersects(exit_point):
                intersection_point = line.interpolate(line.project(exit_point))  # Exact intersection point

    return closest_edge, intersection_point

### split at exit [helper function]

In [21]:
def split_edge_at_exit(G, exit_node, exit_x, exit_y):
    """
    Splits the edge in the graph that is closest to the exit node based on coordinates.
    After splitting, ensures that the new edges have the 'has_exit' attribute if they contain the exit node.
    Also adds the 'has_exit' attribute to the edge if the exit node is already part of it.

    Parameters:
        G (nx.Graph): The NetworkX graph containing the edges to be split.
        exit_node (tuple): The coordinates (x, y) of the exit node to be added.
        exit_x (float): The x-coordinate of the exit point.
        exit_y (float): The y-coordinate of the exit point.

    Actions:
        - Identifies the closest edge to the exit point.
        - Splits the identified edge by adding the exit node and connecting it to both vertices of the edge.
        - Removes the original edge and adds two new edges that include the exit node.
        - Adds the 'has_exit' attribute to the edges, whether split or not.
    """
    # Find the closest edge and the intersection point
    closest_edge, intersection_point = find_closest_edge(G, exit_x, exit_y)

    if closest_edge is None:
        print(f"Warning: No edge found for exit point {exit_node}. Skipping split.")
        return

    u, v = closest_edge

    # If the exit_node is already part of the edge, just mark the edge with 'has_exit' True
    if exit_node == u or exit_node == v:
        print(f"Exit node {exit_node} is already part of the edge ({u}, {v}). Marking it with 'has_exit'.")
        G[u][v]['has_exit'] = True
        
        if exit_node == u:
            G.nodes[u]['is_exit'] = True
        else:
            G.nodes[v]['is_exit'] = True

    # Get attributes from the original edge
    original_slope = G[u][v].get('slope', None)

    # Remove the original edge
    G.remove_edge(u, v)

    # Add the exit node and add the splitted parts of the edge
    if u != exit_node:
        G.add_edge(u, exit_node)
        G[u][exit_node]['has_exit'] = True
        G[u][exit_node]['has_source'] = False
        G[u][exit_node]['slope'] = original_slope
        G[u][exit_node]['edgelength'] = calculate_edge_length(u, exit_node)
    if v != exit_node:
        G.add_edge(exit_node, v)
        G[exit_node][v]['has_exit'] = True
        G[exit_node][v]['has_source'] = False
        G[exit_node][v]['slope'] = original_slope
        G[exit_node][v]['edgelength'] = calculate_edge_length(exit_node, v)

### handling exit points [helper function / mini workflow]

In [22]:
def handle_exit_points(G, exit_points, node_to_stands):
    """
    Processes exit points by adding them as nodes or merging them with existing ones.
    Also updates edge attributes to indicate if they contain an exit node.

    Returns:
        tuple: (count of already contained exit points, count of newly added exit points).
    """
    # Find nearest nodes and determine new exit points
    exitdf = find_nearest_node(G, exit_points)
    exitdf['newexit'] = np.where(exitdf['distance'] <= 10, exitdf['nearest_neighbor_node'], exitdf['exit_point'])

    # Add exit nodes to graph
    for node in exitdf['newexit']:
        G.add_node(node, is_exit=True, is_source=False, stands=node_to_stands.get(node, []))  # Mark node as an exit

    # Split edges at new exit points if necessary
    for _, row in exitdf[exitdf['distance'] > 10].iterrows():
        exit_x, exit_y = row['exit_point']
        closest_edge, intersection_point = find_closest_edge(G, exit_x, exit_y)

        if closest_edge:
            split_edge_at_exit(G, row['newexit'], exit_x, exit_y)

    # Mark edges that contain exit nodes
    exit_nodes = set(exitdf['newexit'])  # Convert to set for fast lookup
    for u, v in G.edges():
        G[u][v]['has_exit'] = u in exit_nodes or v in exit_nodes  

    # Return counts
    return (exitdf['distance'] <= 10).sum(), (exitdf['distance'] > 10).sum(), G

### TO DO: Verify Exit Points (Plot with roads)

## 💾 store updated graph data [helper function]

In [23]:
def store_updated_graph_data(out_path, G, name, prefix):
    """
    Saves the updated graph data (vertices, edges, exit nodes, and source nodes with attributes) into CSV files.

    If there are no exit nodes, the output folder name is prefixed with 'not_processable_'
    and a README.txt is added explaining why it cannot be processed.

    Parameters:
        out_path (str): The directory where the CSV files will be saved.
        G (networkx.Graph): The original undirected graph.
        name (str): The name of the component, used in the file naming.
        prefix (str): A prefix for the file names.
    """

    # --- Check if there are exit nodes
    exit_nodes = [node for node in G.nodes if G.nodes[node].get('is_exit')]

    # --- Adjust out_path if no exit nodes found
    if not exit_nodes:
        base_dir = os.path.dirname(out_path)
        folder_name = os.path.basename(out_path)

        # Only add prefix if folder name itself doesn't start with "not_processable_"
        if not folder_name.startswith("not_processable_"):
            out_path = os.path.join(base_dir, f"not_processable_{folder_name}")

        os.makedirs(out_path, exist_ok=True)

        # --- Create README file
        readme_path = os.path.join(out_path, "README.txt")
        readme_text = (
            f"This component ('{name}') is unconnected to the existing road network.\n\n"
            "It cannot be processed within the current project because the area "
            "between the forest stands from the ZIF and the nearest road segments "
            "belongs to different ownerships.\n\n"
            "Before including this area in future modeling or optimization, it must be clarified "
            "under which conditions and rules road construction would be permitted in those areas."
        )
        with open(readme_path, "w", encoding="utf-8") as f:
            f.write(readme_text)

    # --- File paths
    updated_vertices_file = os.path.join(out_path, f'{prefix}_nodes.csv')
    updated_edges_file = os.path.join(out_path, f'{prefix}_edges.csv')
    updated_edges_with_attributes_file = os.path.join(out_path, f'{prefix}_edges_with_attributes.csv')
    exit_nodes_file = os.path.join(out_path, f'exit_nodes.csv')
    source_nodes_file = os.path.join(out_path, f'source_nodes.csv')

    # --- Save nodes
    vertices_data = [
        {
            'x': node[0],
            'y': node[1],
            'is_exit': G.nodes[node].get('is_exit', False),
            'is_source': G.nodes[node].get('is_source', False),
            'stands': json.dumps(G.nodes[node].get('stands', []))
        }
        for node in G.nodes
    ]
    pd.DataFrame(vertices_data).to_csv(updated_vertices_file, index=False)

    # --- Save exit nodes
    exit_nodes_data = [{'x': n[0], 'y': n[1]} for n in exit_nodes]
    pd.DataFrame(exit_nodes_data).to_csv(exit_nodes_file, index=False)

    # --- Save source nodes
    source_nodes_data = [
        {'x': n[0], 'y': n[1], 'ID_UG': G.nodes[n]['stands']}
        for n in G.nodes if G.nodes[n].get('is_source') is True
    ]
    pd.DataFrame(source_nodes_data).to_csv(source_nodes_file, index=False)

    # --- Save edges
    edges_data = [(f"({u[0]}, {u[1]})", f"({v[0]}, {v[1]})") for u, v in G.edges]
    pd.DataFrame(edges_data, columns=['Node1(x,y)', 'Node2(x,y)']).to_csv(updated_edges_file, index=False)

    # --- Save edges with attributes
    edges_attributes_data = []
    for u, v in G.edges:
        edge_data = {'Node1(x,y)': f"({u[0]}, {u[1]})", 'Node2(x,y)': f"({v[0]}, {v[1]})"}
        edge_data.update(G[u][v])
        edges_attributes_data.append(edge_data)
    pd.DataFrame(edges_attributes_data).to_csv(updated_edges_with_attributes_file, index=False)

    print(f"Graph data saved in: {out_path}")

    return out_path


def store_updated_graph_data(out_path, G, name, prefix):
    """
    Saves the updated graph data (vertices, edges, exit nodes, and source nodes with attributes) into CSV files.

    If there are no exit nodes, the output folder name is prefixed with 'not_processable_'
    and a README.txt is added explaining why it cannot be processed.

    Parameters:
        out_path (str): The directory where the CSV files will be saved.
        G (networkx.Graph): The original undirected graph.
        name (str): The name of the component, used in the file naming.
        prefix (str): A prefix for the file names.
    """

    # --- Check if there are exit nodes
    exit_nodes = [node for node in G.nodes if G.nodes[node].get('is_exit')]

    # --- Adjust out_path if no exit nodes found
    if not exit_nodes:
        # Extract parent directory and rename folder with prefix
        base_dir = os.path.dirname(out_path)
        folder_name = os.path.basename(out_path)
        out_path = os.path.join(base_dir, f"not_processable_{folder_name}")

        os.makedirs(out_path, exist_ok=True)

        # --- Create README file
        readme_path = os.path.join(out_path, "README.txt")
        readme_text = (
            f"This component ('{name}') is unconnected to the existing road network.\n\n"
            "It cannot be processed within the current project because the area "
            "between the forest stands from the ZIF and the nearest road segments "
            "belongs to different ownerships.\n\n"
            "Before including this area in future modeling or optimization, it must be clarified "
            "under which conditions and rules road construction would be permitted in those areas."
        )
        with open(readme_path, "w", encoding="utf-8") as f:
            f.write(readme_text)
    else:
        os.makedirs(out_path, exist_ok=True)

    # --- File paths
    updated_vertices_file = os.path.join(out_path, f'{prefix}_nodes.csv')
    updated_edges_file = os.path.join(out_path, f'{prefix}_edges.csv')
    updated_edges_with_attributes_file = os.path.join(out_path, f'{prefix}_edges_with_attributes.csv')
    exit_nodes_file = os.path.join(out_path, f'exit_nodes.csv')
    source_nodes_file = os.path.join(out_path, f'source_nodes.csv')

    # --- Save nodes
    vertices_data = [
        {
            'x': node[0],
            'y': node[1],
            'is_exit': G.nodes[node].get('is_exit', False),
            'is_source': G.nodes[node].get('is_source', False),
            'stands': json.dumps(G.nodes[node].get('stands', []))
        }
        for node in G.nodes
    ]
    pd.DataFrame(vertices_data).to_csv(updated_vertices_file, index=False)

    # --- Save exit nodes
    exit_nodes_data = [{'x': n[0], 'y': n[1]} for n in exit_nodes]
    pd.DataFrame(exit_nodes_data).to_csv(exit_nodes_file, index=False)

    # --- Save source nodes
    source_nodes_data = [
        {'x': n[0], 'y': n[1], 'ID_UG': G.nodes[n]['stands']}
        for n in G.nodes if G.nodes[n].get('is_source') is True
    ]
    pd.DataFrame(source_nodes_data).to_csv(source_nodes_file, index=False)

    # --- Save edges
    edges_data = [(f"({u[0]}, {u[1]})", f"({v[0]}, {v[1]})") for u, v in G.edges]
    pd.DataFrame(edges_data, columns=['Node1(x,y)', 'Node2(x,y)']).to_csv(updated_edges_file, index=False)

    # --- Save edges with attributes
    edges_attributes_data = []
    for u, v in G.edges:
        edge_data = {'Node1(x,y)': f"({u[0]}, {u[1]})", 'Node2(x,y)': f"({v[0]}, {v[1]})"}
        edge_data.update(G[u][v])
        edges_attributes_data.append(edge_data)
    pd.DataFrame(edges_attributes_data).to_csv(updated_edges_with_attributes_file, index=False)

    print(f"Graph data saved in: {out_path}")


## 4. Calculate the costs associated with each road segment

There are costs associated with the construction, maintenance and upgrade of roads, depending on the slope. They need to be calculated for each road segment according to their length.

We name the 3 categories of slope:
* flat (slope ≤ 5)
* moderate (5 < slope < 25)
* steep (slope ≥ 25)

The costs are fixed values per action and kilometer and slope categorie that when fed as parameters into the model, will be discounted at 3.5% due to the multi-periodidicty of the problem.

### Set base costs

In [24]:
costs = {
    "flat": {  
        "Build5m": 2147,
        "Maintain5m": 1073.5
    },
    "moderate": {
        "Build5m": 0,
        "Maintain5m": 0
    },
    "steep": {
        "Build5m": 7514.5,
        "Maintain5m": 2683.75
    }
}
costs_df = pd.DataFrame(costs).T
print(costs_df)

          Build5m  Maintain5m
flat       2147.0     1073.50
moderate      0.0        0.00
steep      7514.5     2683.75


### approximation for unknown costs

In [25]:
# step 1
costs_df['Build10m'] = 1.5 * costs_df.Build5m
costs_df['Maintain10m'] = costs_df.Maintain5m
costs_df['Upgrade'] = 0.75 * costs_df.Build5m
costs_df

,Build5m,Maintain5m,Build10m,Maintain10m,Upgrade
flat,2147.0,1073.50,3220.50,1073.50,1610.250
moderate,0.0,0.00,0.00,0.00,0.000
steep,7514.5,2683.75,11271.75,2683.75,5635.875


In [26]:
# step 2
costs_df.iloc[1] = (costs_df.iloc[0] + costs_df.iloc[2]) / 2
costs_df

,Build5m,Maintain5m,Build10m,Maintain10m,Upgrade
flat,2147.00,1073.500,3220.500,1073.500,1610.2500
moderate,4830.75,1878.625,7246.125,1878.625,3623.0625
steep,7514.50,2683.750,11271.750,2683.750,5635.8750


In [27]:
costs_df

,Build5m,Maintain5m,Build10m,Maintain10m,Upgrade
flat,2147.00,1073.500,3220.500,1073.500,1610.2500
moderate,4830.75,1878.625,7246.125,1878.625,3623.0625
steep,7514.50,2683.750,11271.750,2683.750,5635.8750


### assign the costs [helper function]

In [28]:
def assign_undiscounted_costs_to_edges(G, name):
    """
    Assign all cost-related variables (Build5m, Maintain5m, Upgrade, Build10m, Maintain10m) to edges
    based on slope, road type, and edge length.
    
    Parameters:
    G (NetworkX graph): The graph representing the road network.
    
    Returns:
    G (NetworkX graph): The graph with updated cost attributes.
    """

    # Iterate over each edge in the graph and set the new attributes
    for u, v, data in G.edges(data=True):
        if 'edgelength' in data and 'slope' in data:  # Ensure both 'edgelength' and 'slope' exist

            # Determine the correct category based on slope
            if data['slope'] <= 5:
                category = "flat"
            elif 5 < data['slope'] < 25:
                category = "moderate"
            else:  # data['slope'] >= 25
                category = "steep"

            # Assign costs based on the selected category
            for column in costs_df.columns:
                data[column] = round(data['edgelength'] * costs_df.loc[category, column])
                
    print(f'{name} costs assigned')
    return G


## 5. Merge edges to improve performance of optimisation algorithms

Note: It would have been possible to merge earlier, but the cost approximization is better this way, because we have more information on the slope if we don't merge the edges before calculation. 

### Rules to merge
1. Maintain boundary intersection nodes
2. Maintain the first exit node after a boundary intersection
(because the closest exit node to an inaccessible stand will always be the first exit in an accessible stand after a boundary crossing)

### merge short edges [helper function] 

In [29]:
def should_skip_or_swap(u, v, G):
    """
    Decide whether to skip an edge between u and v, or swap nodes for processing.

    Returns:
        skip (bool): True if edge should be skipped.
        u, v (nodes): Possibly swapped nodes for consistent processing.
    """

    u_is_exit = G.nodes[u].get('is_exit', False)
    v_is_exit = G.nodes[v].get('is_exit', False)
    u_is_crossing = G.degree[u] > 2
    v_is_crossing = G.degree[v] > 2

    # Case 1: Both crossings
    if u_is_crossing and v_is_crossing:
        return True, u, v  # skip node

    # Case 2: u exit, v crossing
    elif u_is_exit and v_is_crossing:
        return True, u, v  # skip

    # Case 3: u crossing, v exit
    elif u_is_crossing and v_is_exit:
        return True, u, v  # skip
    
    # Case 2: Both u and v are exits
    elif u_is_exit and v_is_exit:
        def has_crossing_neighbor(node):
            return any(G.degree[n] > 2 for n in G.neighbors(node))

        u_has_crossingneighbor = has_crossing_neighbor(u)
        v_has_crossingneighbor = has_crossing_neighbor(v)

        # Both have crossing neighbors → skip
        if u_has_crossingneighbor and v_has_crossingneighbor:
            return True, u, v  # skip edge

        # Only v has crossing neighbors → swap u and v
        elif v_has_crossingneighbor and not u_has_crossingneighbor:
            u, v = v, u
            return False, u, v  # proceed

        # Only u has crossing neighbors → proceed as-is
        elif u_has_crossingneighbor and not v_has_crossingneighbor:
            return False, u, v  # proceed

        # Neither has crossing neighbors → skip
        else:
            return True, u, v  # skip edge

    # Case 5: v is exit or crossing, u normal
    elif v_is_exit or v_is_crossing:
        u, v = v, u  # swap to make v normal
        return False, u, v

    # Case 6: u exit or crossing, v normal
    # and Case 7: both normal
    else:
        return False, u, v  # proceed


In [30]:
def merge_edges(G, length_threshold):
    """
    Merges edges in a graph G that are shorter than a given length_threshold. no exit nodes or crossing can be removed.
    
    Parameters:
    G (networkx.Graph): The input graph containing nodes and edges to be merged.
    length_threshold (float): The maximum length below which edges will be merged.

    Returns:
    networkx.Graph: The modified graph with merged edges.
    """

    while True:
        merged = False  # Flag to track if any merge happens in an iteration

        for u, v, data_edge_uv in list(G.edges(data=True)):

            remove_nodes = []

            #if data_edge_uv.get('edgelength', float('inf')) >= length_threshold:
                #continue  # Skip if the edge is too long
            
            # Call the function to decide skip / swap
            skip, u, v = should_skip_or_swap(u, v, G)
            
            if skip:
                continue  # skip this edge entirely

            other_neighbors_of_v = [n for n in G.neighbors(v) if n != u]

            # Find the shortest edge among v's neighbors
            closest_neighbor_to_v = None
            shortest_length_to_v = float('inf')

            for n in other_neighbors_of_v:
                if G.has_edge(v, n):
                    edge_length = G[v][n].get('edgelength', float('inf'))
                    if edge_length < shortest_length_to_v:
                        shortest_length_to_v = edge_length
                        closest_neighbor_to_v = n

            # Merge the edges if a valid neighbor is found
            if closest_neighbor_to_v is not None and not G.has_edge(u, closest_neighbor_to_v):
                remove_nodes.append(v)
                data_edge_vn = G[v][closest_neighbor_to_v]
    
                # Create merged edge data
                new_data = {
                    'edgelength': data_edge_uv['edgelength'] + data_edge_vn['edgelength'],
                    'has_exit': data_edge_uv.get('has_exit') or data_edge_vn.get('has_exit'),
                    'has_source': False,
                    'Build5m': data_edge_uv.get('Build5m') + data_edge_vn.get('Build5m'),
                    'Maintain5m': data_edge_uv.get('Maintain5m') + data_edge_vn.get('Maintain5m'),
                    'Build10m': data_edge_uv.get('Build10m') + data_edge_vn.get('Build10m'),
                    'Maintain10m': data_edge_uv.get('Maintain10m') + data_edge_vn.get('Maintain10m'),
                    'Upgrade': data_edge_uv.get('Upgrade') + data_edge_vn.get('Upgrade'),
                    'slope' : False,
                    'removed_nodes': data_edge_uv.get('removed_nodes', []) + remove_nodes
                }

                # Add the new merged edge
                G.add_edge(u, closest_neighbor_to_v, **new_data)

                # Remove old edges
                if G.has_edge(u, v):
                    G.remove_edge(u, v)
                if G.has_edge(v, closest_neighbor_to_v):
                    G.remove_edge(v, closest_neighbor_to_v)
                if G.has_node(v):
                    G.remove_node(v)

                merged = True  # Mark that a merge happened
                break  # Restart iteration

        if not merged:
            break  # Stop if no merges happened in this round

    return G

### verify the result of merging [helper function]

In [31]:
def verify_merge(G_before_merge, G_after_merge):
    """
    Verifies the merge by comparing the total sum of `edgelength` and costs 
    before and after the merge.

    Parameters:
        G_before_merge: The graph before merging.
        G_after_merge: The graph after merging.
    """
    def calculate_totals(G):
        return {
            'edgelength': sum(data.get('edgelength', 0) for _, _, data in G.edges(data=True)),
            'Build5m': sum(data.get('Build5m', 0) for _, _, data in G.edges(data=True)),
            'Maintain5m': sum(data.get('Maintain5m', 0) for _, _, data in G.edges(data=True)),
            'Build10m': sum(data.get('Build10m', 0) for _, _, data in G.edges(data=True)),
            'Maintain10m': sum(data.get('Maintain10m', 0) for _, _, data in G.edges(data=True)),
            'Upgrade': sum(data.get('Upgrade', 0) for _, _, data in G.edges(data=True))
        }
    
    totals_before = calculate_totals(G_before_merge)
    totals_after = calculate_totals(G_after_merge)
    
    # Check for differences
    differences = {key: round(totals_after[key] - totals_before[key], 2) for key in totals_before}
    significant_differences = {k: v for k, v in differences.items() if v != 0}
    
    if significant_differences:
        print("Warning: Differences detected in the following totals:")
        for key, value in significant_differences.items():
            print(f"  {key}: {value}")
    
    return {
        'deleted_edges': G_before_merge.number_of_edges() - G_after_merge.number_of_edges()
    }


## 6. Add source nodes

### create source nodes for each stand [helper function]

In [32]:
def add_centroids_and_imaginary_edges(G, stands, exit_points, precision=5):
    """
    Adds center point nodes (centroids) to the graph and creates edges from each 
    centroid to boundary vertices that already exist in the graph, including connecting
    exit nodes to centroids if they lie on the polygon.

    Parameters:
        G (nx.Graph): The NetworkX graph object to which the center points and edges will be added.
        stands (GeoDataFrame): A GeoDataFrame containing the polygon geometries of the stands.
        exit_points (list of tuples): A list of exit points (coordinates where stand boundaries intersect roads).
        precision (int, optional, default=5): The precision to which the coordinates 
                                              will be snapped when extracted (controls decimal places).

    Returns:
        None: Modifies the provided graph in-place by adding new nodes (centroids) and edges.
    """
    for _, feature in stands.iterrows():
        geometry = feature.geometry
        id_ug = feature['ID_UG']  # Extract ID_UG from the stand feature

        # Ensure the geometry is a polygon
        if geometry.geom_type == 'Polygon':
            # Get the boundary coordinates of the polygon (exterior coordinates)
            exterior_coords = [snap_to_grid(coord, precision) for coord in geometry.exterior.coords]

          # Snap the exit points to the grid
            snapped_exit_points = [snap_to_grid(exit_point, precision) for exit_point in exit_points]

            # Check if the snapped exit point is close to the boundary (within a small tolerance)
            tolerance = 1e-5  # Define a tolerance for how close the exit point needs to be to the boundary

            for exit_node in snapped_exit_points:
                exit_point_geom = Point(exit_node)  # Convert the exit node to a Shapely Point geometry

                # Check if the exit point is within the tolerance distance of the polygon boundary
                if geometry.exterior.distance(exit_point_geom) <= tolerance:
                    # Append the exit point to the list of exterior coordinates
                    exterior_coords.append(exit_node)

            
            # Calculate the centroid (center point) of the polygon
            centroid = geometry.centroid
            centroid_coord = snap_to_grid((centroid.x, centroid.y), precision)

            # Add the centroid as a new node to the graph with ID_UG as the source identifier
            G.add_node(centroid_coord, is_exit=False, is_source=True, stands=id_ug)  # Assigning ID_UG instead of True

            # Create edges from the centroid to boundary vertices, only if the boundary vertex is in the graph
            for vertex in exterior_coords:
                if vertex in G:  # Check if the vertex already exists in the graph
                    # Add an edge from the centroid to the boundary vertex
                    if vertex in snapped_exit_points or vertex in exit_points:
                        G.add_edge(centroid_coord, vertex, has_source=True, edgelength=0, slope=False, has_exit=True)
                    else:    
                        G.add_edge(centroid_coord, vertex, has_source=True, edgelength=0, slope=False, has_exit=False)

    return G


### only add source nodes for inaccessible stands

In [33]:
def add_centroids_and_imaginary_edges_for_inaccessiblestands(G, stands, exit_points, precision=5):
    """
    Adds center point nodes (centroids) to the graph and creates edges from each 
    centroid to boundary vertices that already exist in the graph, only for stands
    where road_acces == "none". Also connects exit nodes to centroids if they lie on the polygon.

    Parameters:
        G (nx.Graph): The NetworkX graph object to which the center points and edges will be added.
        stands (GeoDataFrame): A GeoDataFrame containing the polygon geometries of the stands.
        exit_points (list of tuples): A list of exit points (coordinates where stand boundaries intersect roads).
        precision (int, optional, default=5): The precision to which the coordinates 
                                              will be snapped when extracted (controls decimal places).

    Returns:
        nx.Graph: Modified graph with added centroid nodes and edges.
    """
    from shapely.geometry import Point

    for _, feature in stands.iterrows():
        # Skip stands that are accessible
        access = str(feature.get('road_acces')).lower()
        if access != 'none':
            continue  # Only process inaccessible stands

        geometry = feature.geometry
        id_ug = feature['ID_UG']  # Extract ID_UG from the stand feature

        # Ensure the geometry is a polygon
        if geometry.geom_type != 'Polygon':
            continue

        # Get the boundary coordinates of the polygon (exterior coordinates) and snap to grid
        exterior_coords = [snap_to_grid(coord, precision) for coord in geometry.exterior.coords]

        # Snap the exit points to the grid
        snapped_exit_points = [snap_to_grid(exit_point, precision) for exit_point in exit_points]

        # Define a tolerance for how close the exit point needs to be to the boundary
        tolerance = 1e-5

        # Append exit points that lie close to the polygon boundary
        for exit_node in snapped_exit_points:
            exit_point_geom = Point(exit_node)
            if geometry.exterior.distance(exit_point_geom) <= tolerance:
                exterior_coords.append(exit_node)

        # Calculate the centroid of the polygon
        centroid = geometry.centroid
        centroid_coord = snap_to_grid((centroid.x, centroid.y), precision)

        # Add the centroid as a new node to the graph
        G.add_node(centroid_coord, is_exit=False, is_source=True, stands=id_ug)

        # Create edges from the centroid to boundary vertices (only if the boundary vertex exists in the graph)
        for vertex in exterior_coords:
            if vertex in G:
                if vertex in snapped_exit_points or vertex in exit_points:
                    G.add_edge(centroid_coord, vertex, has_source=True, edgelength=0, slope=False, has_exit=True)
                else:
                    G.add_edge(centroid_coord, vertex, has_source=True, edgelength=0, slope=False, has_exit=False)

    return G


### assign costs to source node edges [helper function]

In [34]:
def assign_zero_costs_to_imaginary_edges(G, name):
    """
    Assign zero to all cost-related variables (Build5m, Maintain5m, Upgrade, Build10m, Maintain10m) to edges
    based on slope, road type, and edge length.
    """

    # Iterate over each edge in the graph and set the new attributes
    for u, v, data in G.edges(data=True):
        if 'has_source' in data and data.get('has_source') == True:

            # Assign costs based on the selected category
            for column in costs_df.columns:
                data[column] = 0
                
    print(f'{name} zero costs assigned to imaginary edges')
    return G

## 7. Prepare CPLEX Input

### Create digraph, create arcs, split into subsets [helper function]

In [35]:
def create_digraph_split_arcs(G):
    """
    Splits the arcs of a directed graph (D) into forward and backward arcs based on the original undirected graph (G).

    Parameters:
        D (networkx.DiGraph): The directed graph with bidirectional arcs.
        G (networkx.Graph): The original undirected graph.

    Returns:
        tuple: (forward_arcs, backward_arcs) where each is a list of edges.
    """
    forward_arcs = []
    backward_arcs = []

    # Create a directed graph with bidirectional edges and preserve attributes
    D = nx.DiGraph()

    for u, v, attrs in G.edges(data=True):  # Get attributes from G
        D.add_edge(u, v, **attrs)  # Forward edge with attributes
        D.add_edge(v, u, **attrs)  # Reverse edge with attributes   

    for u, v in G.edges:  # Only loop over edges from G
        if D.has_edge(u, v) and D.has_edge(v, u):  # Ensure both directions exist
            forward_arcs.append((u, v))
            backward_arcs.append((v, u))

    return forward_arcs, backward_arcs, D

### Boundaries

In [36]:
def create_save_boundaries(outpath, G):
    """
    Generates a dictionary that maps each stand (ID_UG) to a list of boundary nodes it contains
    and stores the result in a CSV file with each ID_UG and the list of nodes on its boundaries.

    Parameters:
        outpath (str): The directory path where the output CSV file will be stored.
        G (nx.Graph): The NetworkX graph containing nodes with 'stands' attributes.

    Returns:
        None
    """
    boundaries = {}  # Dictionary to store which nodes are part of each stand
    
    # Iterate over the nodes of the graph
    for node, data in G.nodes(data=True):
        stands = data.get('stands', [])  # List of stand IDs associated with this node
        
        # normalize before loop
        if isinstance(stands, int):
            stands = [stands]
            
        for stand_id in stands:
            # Initialize the list for the stand if not already present
            if stand_id not in boundaries:
                boundaries[stand_id] = []
            
            # Add this node to the list of nodes that touch the stand
            boundaries[stand_id].append(node)

    # Convert the dictionary to a list of dictionaries for easier CSV export
    boundaries_data = []
    for stand, nodes in boundaries.items():
        # Convert the list of nodes to a string representation
        nodes_str = ','.join(map(str, nodes))  # Serialize the list of nodes as a comma-separated string
        boundaries_data.append({'ID_UG': stand, 'nodes': nodes_str})  # Store the list of nodes as a string

    # Define the output file path
    output_file = os.path.join(outpath, "boundaries.csv")

    # Create a DataFrame from the data and save it to a CSV file
    df = pd.DataFrame(boundaries_data)
    df.to_csv(output_file, index=False, sep=';')  # Use semicolon as a separator

    print(f"Stand to nodes data saved to {output_file}")

## Arcs

In [37]:
def create_save_arcs(out_path, G):
    """
    Saves directed arcs, forward arcs, and backward arcs into separate CSV files.

    Parameters:
        out_path (str): The directory where the CSV files will be saved.
        D (networkx.DiGraph): The directed graph containing bidirectional arcs.
        G (networkx.Graph): The original undirected graph.
        name (str): The name of the component, used in the file naming.
        prefix (str): A prefix for the file names.
    """
    # Ensure the folder exists
    os.makedirs(out_path, exist_ok=True)

    # File paths for arcs
    updated_arcs_file = os.path.join(out_path, f'arcs.csv')
    updated_arcs_with_attributes_file = os.path.join(out_path, f'arcs_with_attributes.csv')
    forward_arcs_file = os.path.join(out_path, f'arcsforward.csv')
    backward_arcs_file = os.path.join(out_path, f'arcsbackward.csv')

    # **NEW**: Split arcs into forward and backward
    forward_arcs, backward_arcs, D = create_digraph_split_arcs(G)

    # Extract and save all directed arcs
    arcs_data = [(f"({u[0]}, {u[1]})", f"({v[0]}, {v[1]})") for u, v in D.edges]
    pd.DataFrame(arcs_data, columns=['Source(x,y)', 'Target(x,y)']).to_csv(updated_arcs_file, index=False)

    # Extract and save directed arcs with attributes dynamically
    arcs_attributes_data = []
    for u, v in D.edges:
        arc_data = {'Source(x,y)': f"({u[0]}, {u[1]})", 'Target(x,y)': f"({v[0]}, {v[1]})"}
        arc_data.update(D[u][v])  # Dynamically add all attributes
        arcs_attributes_data.append(arc_data)
    pd.DataFrame(arcs_attributes_data).to_csv(updated_arcs_with_attributes_file, index=False)

    # Save forward arcs
    forward_arcs_data = [(f"({u[0]}, {u[1]})", f"({v[0]}, {v[1]})") for u, v in forward_arcs]
    pd.DataFrame(forward_arcs_data, columns=['Source(x,y)', 'Target(x,y)']).to_csv(forward_arcs_file, index=False)

    # Save backward arcs
    backward_arcs_data = [(f"({u[0]}, {u[1]})", f"({v[0]}, {v[1]})") for u, v in backward_arcs]
    pd.DataFrame(backward_arcs_data, columns=['Source(x,y)', 'Target(x,y)']).to_csv(backward_arcs_file, index=False)

    print(f"Arcs data saved in {out_path}")

In [38]:
def reconcile_folders(out_directory):
    """
    After the main workflow, check for any 'not_processable_*' folders.
    If a corresponding 'comp...' folder exists (matching the suffix),
    move all its contents into the notprocessable folder and delete the original.
    """
    for folder in os.listdir(out_directory):
        if folder.startswith("not_processable_"):
            comp_name = folder.split("not_processable_")[-1]
            not_processable_path = os.path.join(out_directory, folder)
            comp_path = os.path.join(out_directory, comp_name)

            if os.path.exists(comp_path) and os.path.isdir(comp_path):
                print(f"\n[Reconcile] Merging '{comp_name}' → '{folder}'")

                for item in os.listdir(comp_path):
                    src = os.path.join(comp_path, item)
                    dst = os.path.join(not_processable_path, item)

                    # Overwrite existing files or folders
                    if os.path.exists(dst):
                        if os.path.isdir(dst):
                            shutil.rmtree(dst)
                        else:
                            os.remove(dst)

                    shutil.move(src, dst)

                # Remove the now-empty component folder
                shutil.rmtree(comp_path)
                print(f"[Reconcile] Removed original folder: {comp_path}")

## Workflow Step 1 to 7

In [39]:
### WORKFLOW ###
# Define output path for aggregated component information
#base_info_file = f'{out_directory}/info.txt'
#write_base_info_header(base_info_file)  # Write header for summary file

for name, df in components.items():
    print(f"Processing component: {name}")

    # Create a dedicated folder for the current component
    component_folder = f"{out_directory}/{name}"
    os.makedirs(component_folder, exist_ok=True)

    # Extract key graph-related data from the component's dataframe
    boundary_points, boundary_edges, attributes, exit_points, node_to_stands = extract_boundaries_with_attributes(component_folder, df, precision=5)

    # Save the extracted data for further use
    #store_data_for_graph(component_folder, boundary_points, boundary_edges, attributes, name)

    # Construct the initial graph representation
    G = create_graph(boundary_points, boundary_edges, attributes, node_to_stands)

    # Visualize and save the initial graph
    plot_and_save(component_folder, G, name, prefix='0graph')
    save_graph_data_to_csv(component_folder, G, name, prefix='0')

    ### EXIT POINTS ###

    # Process exit points: determine which are contained and which need to be added
    contained_count, to_add_count, G= handle_exit_points(G, exit_points, node_to_stands)
    print(f"{contained_count} exit points already exist")
    print(f"{to_add_count} exit points added")

    # Update and save the plot after handling exit points
    plot_and_save(component_folder, G, name, prefix='1withexits_graph')
    save_graph_data_to_csv(component_folder, G, name, prefix='1withexit')
    
    # Store the updated graph data, including nodes and edges with attributes
    component_folder = store_updated_graph_data(component_folder, G, name, prefix='1withexits')
    print(component_folder)

    #### ASSIGNING COSTS TO EDGES ####
    # Assign cost values to all edges in the graph
    G = assign_undiscounted_costs_to_edges(G, name)

    # Save and visualize the graph with assigned edge costs
    #(component_folder, G, name, prefix='2withcosts')
    plot_and_save(component_folder, G, name, prefix='2withcosts_graph')
    save_graph_data_to_csv(component_folder, G, name, prefix='2withcosts')

    #### MERGING SHORT EDGES ####

    # Create a copy of the graph before merging short edges for comparison
    G_before_merge = G.copy()
        
    # Merge edges that are shorter than the specified length threshold
    G = merge_edges(G,length_threshold=2000) 

    # Visualize and save the updated graph after merging short edges
    plot_and_save(component_folder, G, name, prefix='3baftermerge_graph')
    save_graph_data_to_csv(component_folder, G, name, prefix='3aftermerge')

    # Store the updated graph data with merged edges
    store_updated_graph_data(component_folder, G, name, prefix='3baftermerge')

    verify_merge(G_before_merge, G)
    
    ### SOURCE NODES ###

     # Extract key graph-related data from the component's dataframe
    G = add_centroids_and_imaginary_edges_for_inaccessiblestands(G, df, exit_points, precision=5)

    # assign the zero costs
    G = assign_zero_costs_to_imaginary_edges(G, name)

    # Visualize and save the updated graph after merging short edges
    plot_and_save(component_folder, G, name, prefix='4withsources_graph')
    save_graph_data_to_csv(component_folder, G, name, prefix='4withsources')

    # Store the updated graph data with merged edges
    store_updated_graph_data(component_folder, G, name, prefix='4withsources') 
    create_save_arcs(component_folder, G)

    create_save_boundaries(component_folder, G)

    # Uncomment if you want to write debug information for further analysis
    # write_debug_infos(base_info, component_folder, name)

# Once all components are processed, a summary file containing aggregated information is saved
# print(f"Aggregated information for all components saved in {base_info_file}")


Processing component: comp_10
✅ Graph summary for 'comp_10' appended to 1_Preprocessed_Data\4_Road_Network_Graphs\graph_summary.csv
Exit node (-15124.80997, 154024.99935) is already part of the edge ((-15191.70698, 154023.82671), (-15124.80997, 154024.99935)). Marking it with 'has_exit'.
Exit node (-14857.24072, 153996.39466) is already part of the edge ((-14841.81778, 153992.33571), (-14857.24072, 153996.39466)). Marking it with 'has_exit'.
Exit node (-15032.51193, 154004.65843) is already part of the edge ((-15014.18668, 154002.06851), (-15032.51193, 154004.65843)). Marking it with 'has_exit'.
16 exit points already exist
23 exit points added
✅ Graph summary for 'comp_10' appended to 1_Preprocessed_Data\4_Road_Network_Graphs\graph_summary.csv
Graph data saved in: 1_Preprocessed_Data\4_Road_Network_Graphs/comp_10
1_Preprocessed_Data\4_Road_Network_Graphs/comp_10
comp_10 costs assigned
✅ Graph summary for 'comp_10' appended to 1_Preprocessed_Data\4_Road_Network_Graphs\graph_summary.csv

In [40]:
# Number of rows in the dataframe
print(len(stands))

# Number of unique IDs
print(stands['ID_UG'].nunique())

# Optional: see duplicates
duplicates = stands[stands.duplicated(subset='ID_UG', keep=False)]
print('duplicates:', duplicates)

20
20
duplicates: Empty GeoDataFrame
Columns: [OBJECTID, TARGET_FID, LandUse_1, Ocupacao, ID_UG, NOME, YY_correct, XX_Correct, Altitude, Declive, Litologia, Relevo, Uso, Espessura, Apt_flor, Solo, Classifica, Perimetro, Area, Hectares, UsoSolo20, Shape_Leng, Shape_Area, HBC, CH, CBD, CC, road_acces, geometry]
Index: []

[0 rows x 29 columns]


## cleanup directory to store unprocessable component separately

In [41]:
# --- Run the cleanup after all components are processed ---
reconcile_folders(out_directory)
print("\nAll components processed and notprocessable folders reconciled.")


[Reconcile] Merging 'comp_17' → 'not_processable_comp_17'
[Reconcile] Removed original folder: 1_Preprocessed_Data\4_Road_Network_Graphs\comp_17

All components processed and notprocessable folders reconciled.


## summarize and aggregate
take out components that cant be solved in this work

In [47]:
out_dir = Path(out_directory)
summary_path = out_dir / "graph_summary.csv"
df = pd.read_csv(summary_path)

In [42]:
def compare_graph_summaries(out_directory):
    """
    Reads the combined 'graph_summary.csv' file in 'out_directory',
    aggregates totals for each Prefix, and saves two comparison tables:
    1. network_characteristics_comparison.csv (all components)
    2. network_characteristics_comparison_onlyprocessablecomponents.csv (without non-processable components)

    Parameters:
        out_directory (str): Directory containing 'graph_summary.csv'.
    """

    out_dir = Path(out_directory)
    summary_path = out_dir / "graph_summary.csv"

    if not summary_path.exists():
        print(f"⚠️ No 'graph_summary.csv' found in {out_directory}")
        return

    df = pd.read_csv(summary_path)

    # Ensure numeric conversion for key columns
    numeric_cols = [
        "NumNodes",
        "NumEdges",
        "NumSourceNodes",
        "NumExitNodes",
        "TotalEdgeLength",
        "TotalCosts",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "Prefix" not in df.columns:
        print("⚠️ 'Prefix' column missing in CSV — cannot aggregate.")
        return

    # ------------------------------------------------------------------
    # Comparison for ALL components
    # ------------------------------------------------------------------
    comparison_all = (
        df.groupby("Prefix", as_index=False)
        .agg({
            "NumNodes": "sum",
            "NumEdges": "sum",
            "NumSourceNodes": "sum",
            "NumExitNodes": "sum",
            "TotalEdgeLength": "sum",
            "TotalCosts": "sum" if "TotalCosts" in df.columns else "first",
        })
        .rename(columns={
            "NumNodes": "Total_NumNodes",
            "NumEdges": "Total_NumEdges",
            "NumSourceNodes": "Total_SourceNodes",
            "NumExitNodes": "Total_ExitNodes",
            "TotalEdgeLength": "Total_EdgeLength",
            "TotalCosts": "Total_Costs",
        })
        .sort_values(by="Prefix", ignore_index=True)
    )

    output_all = out_dir / "network_characteristics_comparison.csv"
    comparison_all.to_csv(output_all, index=False)
    print(f"✅ Comparison table (all components) saved to: {output_all}")

    # ------------------------------------------------------------------
    # Identify non-processable components from folder names
    # ------------------------------------------------------------------
    not_processable_components = {
        p.name.removeprefix("not_processable_")
        for p in out_dir.iterdir()
        if p.is_dir() and p.name.startswith("not_processable_")
    }
    
    if not_processable_components:
        print('not processable:', not_processable_components)
    else:
        print("ℹ️ No non-processable component folders found.")

    # ------------------------------------------------------------------
    # Filter out non-processable components
    # ------------------------------------------------------------------
    if "ComponentName" not in df.columns:
        print("⚠️ 'ComponentName' column not found in CSV — skipping filtered comparison.")
        return comparison_all

    # Use exact match (fast & safe)
    df_filtered = df[
        ~df["ComponentName"].isin(not_processable_components)
    ].copy()

    filtered_summary_path = out_dir / "graph_summary_onlyprocessablecomponents.csv"
    df_filtered.to_csv(filtered_summary_path, index=False)
    print(f"✅ Filtered summary saved to: {filtered_summary_path}")
    print(f"🗑️ Removed {len(df) - len(df_filtered)} rows with non-processable components.")

    # ------------------------------------------------------------------
    # Comparison for ONLY processable components
    # ------------------------------------------------------------------
    comparison_filtered = (
        df_filtered.groupby("Prefix", as_index=False)
        .agg({
            "NumNodes": "sum",
            "NumEdges": "sum",
            "NumSourceNodes": "sum",
            "NumExitNodes": "sum",
            "TotalEdgeLength": "sum",
            "TotalCosts": "sum" if "TotalCosts" in df_filtered.columns else "first",
        })
        .rename(columns={
            "NumNodes": "Total_NumNodes",
            "NumEdges": "Total_NumEdges",
            "NumSourceNodes": "Total_SourceNodes",
            "NumExitNodes": "Total_ExitNodes",
            "TotalEdgeLength": "Total_EdgeLength",
            "TotalCosts": "Total_Costs",
        })
        .sort_values(by="Prefix", ignore_index=True)
    )

    output_filtered = out_dir / "network_characteristics_comparison_onlyprocessablecomponents.csv"
    comparison_filtered.to_csv(output_filtered, index=False)
    print(f"✅ Comparison table (only processable components) saved to: {output_filtered}")

    return comparison_all, comparison_filtered


In [43]:
all, without_notprocessable = compare_graph_summaries(out_directory)

✅ Comparison table (all components) saved to: 1_Preprocessed_Data\4_Road_Network_Graphs\network_characteristics_comparison.csv
not processable: {'comp_17'}
✅ Filtered summary saved to: 1_Preprocessed_Data\4_Road_Network_Graphs\graph_summary_onlyprocessablecomponents.csv
🗑️ Removed 5 rows with non-processable components.
✅ Comparison table (only processable components) saved to: 1_Preprocessed_Data\4_Road_Network_Graphs\network_characteristics_comparison_onlyprocessablecomponents.csv


In [44]:
all

,Prefix,Total_NumNodes,Total_NumEdges,Total_SourceNodes,Total_ExitNodes,Total_EdgeLength,Total_Costs
0,0,16787,17357,0,0,759.98,0.0
1,1withexit,17218,17788,0,736,759.98,0.0
2,2withcosts,17218,17788,0,736,759.98,3267899.0
3,3aftermerge,1533,2103,0,448,759.98,3267899.0
4,4withsources,1848,3741,315,448,759.98,3267899.0


In [45]:
without_notprocessable

,Prefix,Total_NumNodes,Total_NumEdges,Total_SourceNodes,Total_ExitNodes,Total_EdgeLength,Total_Costs
0,0,16729,17299,0,0,759.19,0.0
1,1withexit,17160,17730,0,736,759.19,0.0
2,2withcosts,17160,17730,0,736,759.19,3263460.0
3,3aftermerge,1530,2100,0,448,759.19,3263460.0
4,4withsources,1844,3735,314,448,759.19,3263460.0
